# 第三部分：工具人代理 (Action & Agency)
### *Alex 的进化：从建议到执行*

在 **Notebook 2** 中，AI 像个老教授，只给建议不动手。Alex 还是得自己去处理那堆讨厌的 `$` 符号。  
今天，我们将使用 **DeepAgent** 的核心模式：为智能体配置 **工具 (Tools)**。  

在本节课中，你将学习：
1. **ReAct 智能体架构**：理解 LLM 如何通过“推理-行动-观察”循环来解决问题。
2. **集成 Python REPL 工具**：赋予 AI 动态编写并运行代码的能力。
3. **闭环自动化**：观察 AI 如何自主修复我们在 Notebook 1 中遇到的 CSV 崩溃问题。

In [1]:
import os
import pandas as pd
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_experimental.utilities import PythonREPL
from langchain.agents import create_agent 
from langchain.chat_models import init_chat_model

load_dotenv(override=True)

# 使用最新的 init_chat_model 方式
model = init_chat_model(
    model="agnes-2.0-flash",
    model_provider="openai",
    base_url=os.getenv("AGNES_BASE_URL"),
    api_key=os.getenv("AGNES_API_KEY"),
)

print("模型初始化成功。")

模型初始化成功。


## 1. 定义 Python 分析工具
在 DeepAgent 模式下，工具的 `docstring` (文档字符串) 是最重要的。LLM 正是通过阅读它来决定何时调用。

In [2]:
repl = PythonREPL()

@tool
def python_analyst(code: str):
    """
    这是一个 Python 代码执行器。当需要清洗 CSV 数据、修改 DataFrame、处理复杂的财务公式
    或生成分析图表时，请调用此工具。你可以通过它直接操作本地文件系统中的数据。
    输入应该是合法的 Python 代码字符串。

    注意：你可以直接访问 './enterprise_data/' 目录下的文件。
    输入应该是合法的 Python 代码。
    """

    try:
        return repl.run(code)
    except Exception as e:
        return f"执行错误: {e}"

tools = [python_analyst]
print("分析工具已准备就绪。")

分析工具已准备就绪。


## 2. 构建 ReAct 智能体
我们将使用 `create_agent`。这是目前 LangGraph/DeepAgent 推荐的构建方式，它原生支持消息列表和状态管理。

In [3]:
system_message = """
你是一位 GlobalCorp 的资深数据分析助手。
你有权限使用 python_analyst 工具来处理本地数据。
记得在处理现代市场数据 (modern_marketing.csv) 时，必须将其清洗为财务部要求的格式：
- 将 'market' 视为 'Region'
- 将 'qty' 视为 'Quantity'
- 必须清除 'price_per_unit' 中的美元符号并转为数字作为 'Unit_Price'
"""

agent = create_agent(
    model,
    tools=tools,
    system_prompt=system_message
)

print("Agent 编译完成。")

Agent 编译完成。


## 3. 运行自动化任务：自主修复崩溃
我们给 Agent 下达指令，看它如何自主地观察错误、编写修复代码并给出答案。

In [4]:
from utils import format_messages # 假设你本地有这个格式化函数

query = """
请读取 'enterprise_data/modern_marketing.csv' 文件，
按照财务部标准清洗数据，并计算每个 Region 的总收入是多少？
"""

inputs = {"messages": [("user", query)]}

print("--- Agent 开始思考并行动 ---\n")
config = {"recursion_limit": 20}
result = agent.invoke(inputs, config=config)

# 打印最后的 AI 消息内容
print("\n--- 最终结果 ---\n")
print(result["messages"][-1].content)

--- Agent 开始思考并行动 ---



Python REPL can execute arbitrary code. Use with caution.



--- 最终结果 ---

根据财务部要求清洗数据并计算后，每个 **Region** 的总收入如下：

| Region | Total Revenue |
| :--- | :--- |
| EMEA | 2,500.00 |
| LATAM | 2,100.00 |
| US | 10,000.00 |

### 处理步骤摘要：
1. **读取数据**：从 `enterprise_data/modern_marketing.csv` 加载数据。
2. **列名映射**：
   - `market` → `Region`
   - `qty` → `Quantity`
   - `price_per_unit` → `Unit_Price`
3. **数据清洗**：
   - 从 `Unit_Price` 中移除美元符号 `$` 和千位分隔符 `,`。
   - 将清洗后的字符串转换为浮点数。
4. **计算收入**：`Total Revenue = Quantity * Unit_Price`。
5. **汇总**：按 `Region` 分组求和，得出各地区的总收入。


## 4. 复盘：代理的“局限性”

你会发现，Agent 现在表现得像个神童：它自己写了代码，自己修复了 CSV 格式，甚至输出了结果。

### 但是，Alex 发现了一个新的危机：
1. **指令冗余**：Alex 发现在 `system_message` 里写规则非常累。如果公司有 100 种不同的 Excel 格式，这个 Prompt 会变得无限长。
2. **可维护性差**：如果税务规则变了，Alex 必须在代码里修改 System Prompt。这不像是一个成熟系统的做法。
3. **性能衰减**：随着规则（提示词）越来越多，Agent 可能会开始混淆 Sales 和 Marketing 的规则。

### 核心痛点
当“企业规则”堆积如山时，**“超级提示词 (Mega-Prompt)”** 会导致模型认知负荷过重。

**在接下来的 Notebook 4 中，我们将尝试挑战这种“Mega-Prompt”的极限。我们要模拟把 10 个部门的规则都塞进去，看看 Agent 是如何在复杂性中“迷失”并开始犯错的。这将直接引出我们最终的解决方案：DeepAgent Skills。**